# Part 1 — promising directions that did not improve the winner

We keep several distinct failed directions rather than every sweep: log-odds pooling, Other-label specialists and learned combination, partial-label treatment, a diversity-penalized kernel head, and symbolic regression. These explain what we tried to add and why the .89935 model remained selected.

This notebook independently rescores retained useful predictions and PySR sufficient statistics. It does not claim new full fitting of all historical variants. Full compact result tables retain unfavorable results, and notebook 05 exposes the model/loss/training source. Fresh winning-model training is in notebook 01; symbolic search requires its separately recorded Python/Julia environment.
Run with Python 3.12 and the pinned `requirements.txt` beside this notebook. In Colab, extract the complete ZIP, change into `midterm/part1`, and install those dependencies before running. The supplied inputs and selected evidence are included; no Git checkout, private path, external trial data, or model cache is required. Notebook 05 contains the complete packaged Python source.

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits

# Works from the extracted part1 directory or from the repository root.
options = [Path.cwd(), Path.cwd() / "submission/midterm/part1", Path.cwd() / "midterm/part1"]
HOME = next((p for p in options if (p / "resources/handin.py").is_file()), None)
if HOME is None:
    raise RuntimeError("Extract the complete Part 1 ZIP and open this notebook beside its resources folder.")
RES = HOME / "resources"
OUT = HOME / "generated"
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(RES))
import handin as h
P1, LABELS, EMB = h.P1, h.LABELS, h.EMB
# Stable numerical behavior; release automatically when the kernel exits.
thread_limit = threadpool_limits(limits=1)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
display(h.verify_resources())

{'files': 763, 'hashes_match': True, 'external_checkout_required': False}

## Arithmetic versus log-odds pooling

The arithmetic parent mixes raw anchored scores and Cand2 marginals at .8/.2. The refinement kept those weights but pooled log-odds. Small local gains did not translate: the reported score was **.88888**. Changing score combination is not guaranteed to preserve useful calibration or decision margins.

In [2]:
refinement=pd.read_csv(P1 / "experiments/blend_refinement_v2/results/summary.csv")
display(refinement[refinement.recipe.isin(["parent","log_pool"])])

,pool,recipe,repeats,mean,population_sd,macro_f1,delta,wins,ties,losses
2,chronology,log_pool,1,0.887755,0.000000,0.839773,0.000000,0,1,0
5,chronology,parent,1,0.887755,0.000000,0.842570,0.000000,0,1,0
9,pooled,log_pool,8,0.922429,0.006993,0.853693,0.000798,5,2,1
12,pooled,parent,8,0.921631,0.005891,0.851670,0.000000,0,8,0
16,train,log_pool,8,0.935775,0.006693,0.897491,0.000531,3,4,1
19,train,parent,8,0.935244,0.006457,0.895350,0.000000,0,8,0
23,validation,log_pool,1,0.891026,0.000000,0.765430,0.000000,0,1,0
26,validation,parent,1,0.891026,0.000000,0.765430,0.000000,0,1,0


## Other specialists, pseudo-label weighting and learned combination

Other is a weak label: only ten validation positives. We tested nonlinear Other specialists, altered pseudo-label weighting, and nested learned heads. These recipes had to outperform the parent across both grouped pools. None qualified. Learned fusion used inner held-out inputs; it was not fitted to the same outer predictions used to evaluate it.

Partial-label experiments treated absent pseudo-Other as unknown rather than negative, while preserving supplied negative labels. A matched complete-label binary control distinguishes the architecture change from the unknown-label hypothesis. The submitted variant scored **.86645**. Its improved chronology did not justify a general claim of better performance.

In [3]:
checks=[]
for family in ["other_combiner_v2","partial_secondary_v1"]:
    rescored=h.compact_rescore(f"research/{family}/results/retained_comparison.npz")
    original=pd.read_csv(P1 / "research" / family / "results/summary/repeat_scores.csv")
    merged=rescored.merge(original,on=["pool","seed","recipe"],suffixes=("_fresh","_recorded"),validate="one_to_one")
    assert len(merged)==63
    np.testing.assert_allclose(merged.sample_f1_fresh,merged.sample_f1_recorded,atol=1e-12,rtol=0)
    np.testing.assert_allclose(merged.macro_f1_fresh,merged.macro_f1_recorded,atol=1e-12,rtol=0)
    checks.append({"family":family,"rescored_rows":len(merged)})
    print(family,"— all compact variant summaries")
    overview=pd.read_csv(P1 / "research" / family / "results/summary/overview.csv")
    display(overview[overview.pool.isin(["train","pooled","validation","chronology98"])].pivot(index="recipe",columns="pool",values="mean_sample_f1"))

other_combiner_v2 — all compact variant summaries


pool,chronology98,pooled,train,validation
recipe,,,,
anchored,0.884354,0.921365,0.932059,0.891026
calibration_control,0.884354,0.921720,0.934979,0.891026
combiner_other_only,0.891156,0.921631,0.935244,0.891026
combiner_per_label,0.884354,0.922252,0.934979,0.891026
other_conditional_rbf,0.891156,0.921099,0.934713,0.891026
other_rbf,0.891156,0.921631,0.934183,0.891026
parent,0.887755,0.921631,0.935244,0.891026
pseudo_lda_confidence,0.874150,0.921897,0.934713,0.886752
pseudo_lda_quarter,0.874150,0.921897,0.934713,0.886752


partial_secondary_v1 — all compact variant summaries


pool,chronology98,pooled,train,validation
recipe,,,,
anchored,0.884354,0.921365,0.932059,0.891026
binary_complete,0.885714,0.919681,0.931794,0.882479
binary_complete_half,0.885714,0.917553,0.933652,0.882479
binary_unknown,0.901020,0.914982,0.920382,0.891026
binary_unknown_half,0.892517,0.917376,0.928875,0.878205
neighbor_unknown,0.885714,0.919149,0.932856,0.891026
parent,0.887755,0.921631,0.935244,0.891026


## Explicitly complementary kernel head and learned fusion

The second head is a classical Nyström RBF feature model trained to predict labels while penalizing squared correlation between its residuals and the anchor's inner out-of-fold residuals. The ordinary-head control separates kernel capacity from the diversity term. We tested fixed score mixing and learned fusion. This was not joint neural-head training or removal of neural layers.

A stronger penalty improved the fixed blend relative to its ordinary-head control in both grouped pools, but no variant displaced the .89935 parent. The submitted fixed blend scored **.89316**. Because the ordinary-kernel blend was not submitted, this does not isolate the penalty's Kaggle effect.

In [4]:
z=h.arrays(P1 / "research/complementary_heads_v2/results/matched_predictions.npz")
reference=pd.read_csv(P1 / "research/complementary_heads_v2/results/summary/repeat_scores.csv")
records=[]
for pool,seed in reference[["pool","seed"]].drop_duplicates().itertuples(index=False,name=None):
    if pool in ["train","pooled"]:
        keep=np.char.startswith(z["case"],f"{pool}_{int(seed)}_")
    elif pool=="chronology98":keep=np.char.startswith(z["case"],"through_")
    else:keep=z["case"]==pool
    assert keep.any(),pool
    for j,name in enumerate(z["names"]):
        metric,_=h.metrics(z["truth"][keep],z["predictions"][j,keep])
        records.append({"pool":pool,"seed":seed,"recipe":name,**metric})
rescored=pd.DataFrame(records)
merged=rescored.merge(reference,on=["pool","seed","recipe"],suffixes=("_fresh","_recorded"),validate="one_to_one")
assert len(merged)==189
np.testing.assert_allclose(merged.sample_f1_fresh,merged.sample_f1_recorded,atol=1e-12,rtol=0)
np.testing.assert_allclose(merged.macro_f1_fresh,merged.macro_f1_recorded,atol=1e-12,rtol=0)
checks.append({"family":"complementary_heads_v2","rescored_rows":len(merged)})
overview=pd.read_csv(P1 / "research/complementary_heads_v2/results/summary/overview.csv")
display(overview.pivot(index="recipe",columns="pool",values="mean_sample_f1"))

pool,chronology98,pooled,through_2004_test_2005_2006,through_2006_test_2007_2007,through_2007_test_2008_2010,train,validation
recipe,,,,,,,
anchored,0.884354,0.921365,0.863248,0.939394,0.873874,0.932059,0.891026
blend_diverse10,0.874150,0.920567,0.863248,0.893939,0.873874,0.933386,0.886752
blend_diverse30,0.870748,0.920922,0.863248,0.878788,0.873874,0.933386,0.886752
blend_ordinary,0.877551,0.919681,0.863248,0.893939,0.882883,0.931529,0.886752
calibration,0.870748,0.918883,0.829060,0.939394,0.873874,0.932059,0.882479
fusion_diverse10,0.874150,0.919504,0.829060,0.939394,0.882883,0.928079,0.880342
fusion_diverse30,0.874150,0.919326,0.829060,0.939394,0.882883,0.927813,0.880342
fusion_ordinary,0.870748,0.919504,0.829060,0.924242,0.882883,0.929671,0.880342
head_diverse10,0.804422,0.870301,0.743590,0.825758,0.855856,0.902760,0.839744


## Bounded symbolic regression (PySR)

PySR learned small expressions over neighbor, centroid and cluster signals, with unsupervised label generation. The fixed search covered all 84 cases and five label heads per case. Its declared parent was the earlier **.89850 anchored75/LDA25** blend. Both standalone PySR and its fixed 20% contribution regressed on all four headline checks. The exploratory blend scored **.89423**.

The following code rebuilds repeat sample-F1 and per-label/macro F1 from retained sums and confusion counts. Symbolic searches are stochastic software-dependent computations; this package retains the search code, fixed settings, equations and Python/Julia locks, while default notebook execution does not install Julia or launch 420 searches.

In [5]:
stats=pd.read_csv(P1 / "research/pysr_v1/results/sufficient_statistics.csv")
records=[]
for (mode,pool,seed),g in stats[stats.pool.isin(["train","pooled"])].groupby(["mode","pool","seed"]):
    sums=g.select_dtypes(include=[np.number]).sum()
    fs=[]
    for label in LABELS:
        tp,fp,fn=[sums[f"{key}_{label}"] for key in ["tp","fp","fn"]]
        fs.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)
    records.append({"mode":mode,"pool":pool,"seed":seed,"sample_f1":sums.sample_f1_sum/sums.rows,"macro_f1":np.mean(fs)})
repeats=pd.DataFrame(records)
summary=repeats.groupby(["mode","pool"]).sample_f1.mean().unstack()
reference=pd.read_csv(P1 / "research/pysr_v1/results/cv_summary.csv")
for row in reference.itertuples():
    assert abs(summary.loc[row.mode,row.pool]-row.mean_sample_f1)<1e-12
display(summary, pd.read_csv(P1 / "research/pysr_v1/results/supplementary_scores.csv"))
display(pd.read_csv(P1 / "research/pysr_v1/results/equations.csv").head(10))
checks.append({"family":"pysr_v1","rescored_rows":len(repeats)})
pd.DataFrame(checks).to_csv(OUT / "negative_evidence_checks.csv",index=False)

pool,pooled,train
mode,,
anchored,0.921365,0.932059
anchored80_cand2_20,0.921631,0.935244
current75_25,0.923404,0.932059
current80_pysr20,0.919770,0.927548
linear_geometry,0.883564,0.884820
pysr,0.884184,0.869559


,mode,case,sample_f1,macro_f1,precision_Safety,precision_Funding,precision_Efficacy,precision_Recruitment,precision_Other,recall_Safety,...,f1_Safety,f1_Funding,f1_Efficacy,f1_Recruitment,f1_Other,support_Safety,support_Funding,support_Efficacy,support_Recruitment,support_Other
0,anchored,through_2004_test_2005_2006,0.863248,0.809231,1.000000,0.750000,0.769231,0.894737,0.625000,0.750000,...,0.857143,0.750000,0.869565,0.944444,0.625000,4.0,4.0,10.0,17.0,8.0
1,anchored,through_2006_test_2007_2007,0.939394,0.762393,1.000000,0.000000,1.000000,0.857143,1.000000,1.000000,...,1.000000,0.000000,1.000000,0.923077,0.888889,7.0,0.0,6.0,6.0,5.0
2,anchored,through_2007_test_2008_2010,0.873874,0.826450,0.857143,1.000000,0.833333,0.937500,0.666667,0.857143,...,0.857143,0.888889,0.833333,0.937500,0.615385,7.0,5.0,6.0,16.0,7.0
3,anchored,validation,0.891026,0.765430,1.000000,1.000000,0.733333,0.980392,0.571429,0.888889,...,0.941176,0.666667,0.758621,0.990099,0.470588,9.0,4.0,14.0,50.0,10.0
4,anchored,chronology98,0.884354,0.841633,0.941176,0.777778,0.840000,0.904762,0.722222,0.888889,...,0.914286,0.777778,0.893617,0.938272,0.684211,18.0,9.0,22.0,39.0,20.0
5,current75_25,through_2004_test_2005_2006,0.863248,0.809231,1.000000,0.750000,0.769231,0.894737,0.625000,0.750000,...,0.857143,0.750000,0.869565,0.944444,0.625000,4.0,4.0,10.0,17.0,8.0
6,current75_25,through_2006_test_2007_2007,0.939394,0.762393,1.000000,0.000000,1.000000,0.857143,1.000000,1.000000,...,1.000000,0.000000,1.000000,0.923077,0.888889,7.0,0.0,6.0,6.0,5.0
7,current75_25,through_2007_test_2008_2010,0.873874,0.826450,0.857143,1.000000,0.833333,0.937500,0.666667,0.857143,...,0.857143,0.888889,0.833333,0.937500,0.615385,7.0,5.0,6.0,16.0,7.0
8,current75_25,validation,0.891026,0.754973,0.888889,1.000000,0.733333,0.980392,0.571429,0.888889,...,0.888889,0.666667,0.758621,0.990099,0.470588,9.0,4.0,14.0,50.0,10.0
9,current75_25,chronology98,0.884354,0.841633,0.941176,0.777778,0.840000,0.904762,0.722222,0.888889,...,0.914286,0.777778,0.893617,0.938272,0.684211,18.0,9.0,22.0,39.0,20.0


,case,label,expression,complexity,loss,constant_fallback
0,train_20261126_0,Safety,centroid_margin*neighbor_logodds*neighbor_logo...,9,0.078915,False
1,train_20261126_0,Funding,neighbor_logodds*6.067525276562805 - 20.898234...,5,0.022167,False
2,train_20261126_0,Efficacy,cluster_logodds - 7.141009321226055*rival_neig...,7,0.059149,False
3,train_20261126_0,Recruitment,neighbor_logodds*5.380604827968468 + 2.4191942...,5,0.119560,False
4,train_20261126_0,Other,neighbor_logodds - 1.8040297496095585,3,0.349524,False
5,train_20261126_1,Safety,centroid_margin*neighbor_logodds*neighbor_logo...,7,0.066970,False
6,train_20261126_1,Funding,neighbor_logodds*2.799133817002653 - 1*10.1093...,5,0.038387,False
7,train_20261126_1,Efficacy,neighbor_logodds*3.118937655674973 - 3.9507605...,5,0.129040,False
8,train_20261126_1,Recruitment,cluster_logodds*3.9582760632045395,3,0.167132,False
9,train_20261126_1,Other,neighbor_logodds - 1*1.9704605465704965,3,0.344412,False


## What we learned

A small component can help despite being worse alone, as Cand2 did. More complexity, explicit diversity, a learned combiner or better chronology is not sufficient evidence of a better hidden-test model. Preserve matched controls, look across metrics without combining incompatible protocols, and retain negative outcomes. These comparisons repeatedly reuse development labels; no overlapping-repeat significance claim is made.